<a href="https://colab.research.google.com/github/BinduPranaya/Interview-Coach-AI-Communication-Confidence-Analyzer/blob/main/My_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STEP 0: Install dependencies

In [2]:
!pip install -qU google-generativeai==0.8.5 google-ai-generativelanguage==0.6.15 langgraph langchain langchain-google-genai openai

# STEP 1: Imports and secure API key input

In [3]:
import os
import getpass
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage


# Secure Gemini API Key input

In [7]:
os.environ["GOOGLE_API_KEY"]= getpass.getpass("Enter your api key")

Enter your api key··········


# STEP 2: Initialize Gemini 2.5 Flash

In [8]:
llm = ChatGoogleGenerativeAI( model = "models/gemini-2.5-flash", temperature = 0.2)

# STEP 3: Node to ask user for symptom

In [9]:
def get_symptom(state: dict)-> dict:
  symptom = input("Welcome to Evergreen Hospital, Please Enter your symptoms")
  state["symptom"] = symptom
  return state

# STEP 4: Node to classify the symptom

In [10]:
def classify_symptom(state: dict)-> dict:
  prompt = (
      "You are a helpful medical assistant, classify the input as"
      "-general, Emergency, mental health"
      f"symptom L{state['symptom']}"
      'respond only with one word: general, emergency or mental'
      "#Example: input : I have fever, output: general"
  )
  response = llm.invoke([HumanMessage(content=prompt)])
  category = response.content.strip()
  print(f"LLM Classify the symptom as : {category}") #debug
  state["category"] = category
  return state

# STEP 5: Router logic to route to the correct node


In [11]:
def symptom_router(state:dict)->dict:
  cat = state["category"].lower()
  if "general" in cat:
    return "general"
  elif "emergency" in cat:
    return "emergency"
  elif "mental" in cat:
    return "mental_health"
  else:
    return "general"

# STEP 6: Category-specific response nodes


In [12]:
def general_node(state: dict) -> dict:
  state["answer"] = f" '{state['symptom']}' : seems general : directing you to general ward for consulting a doctor"
  return state

def emergency_node(state:dict) -> dict:
  state["answer"] =f" ' {state['symptom']}': It is a Medical Emergency : seeking immediate help"
  return state

def mental_health_node(state: dict) -> dict:
  state["answer"] = f" '{state['symptom']}: seems like a medical health issue: talk to our counsellor"
  return state

# STEP 7: Build LangGraph


In [13]:
builder = StateGraph(dict)

#define the nodes
builder.set_entry_point("get_symptom")
builder.add_node("get_symptom", get_symptom)
builder.add_node("classify", classify_symptom)
builder.add_node("general", general_node)
builder.add_node("emergency", emergency_node)
builder.add_node("mental_health", mental_health_node)

builder.add_edge("get_symptom", "classify")
builder.add_conditional_edges("classify", symptom_router, {
    "general": "general",
    "emergency": "emergency",
    "mental_health": "mental_health"
})

builder.add_edge("general", END)
builder.add_edge("emergency", END)
builder.add_edge("mental_health", END)

# STEP 8: Compile and invoke the graph


In [14]:
graph = builder.compile()

In [15]:
final_state = graph.invoke({})
print("final Output \n")
print(final_state["answer"])

Welcome to Evergreen Hospital, Please Enter your symptomsi am feeling headache and high temperature in my body.
LLM Classify the symptom as : general
final Output 

 'i am feeling headache and high temperature in my body.' : seems general : directing you to general ward for consulting a doctor
